# Step 2: Extract HKS Features for Labeled Spines

Now that you have labeled spines in Napari and saved them to `spine_labels_v1.csv`, we need to compute the mathematical features (**HKS**) for those specific locations.

In [ ]:
import os
import pandas as pd
import numpy as np
import caveclient
from meshparty import trimesh_io
from meshmash import condensed_hks_pipeline
from tqdm.notebook import tqdm

# Constants
DATASTACK = 'minnie65_public'
ANNOTATION_FILE = 'spine_labels_v1.csv'
CACHE_DIR = 'meshes'

client = caveclient.CAVEclient(DATASTACK)
mm = trimesh_io.MeshMeta(cv_path=client.info.segmentation_source(), disk_cache_path=CACHE_DIR)

In [ ]:
if not os.path.exists(ANNOTATION_FILE):
    raise FileNotFoundError(f"{ANNOTATION_FILE} not found. Please ensure you CLOSED the Napari window to save your points.")

df_labels = pd.read_csv(ANNOTATION_FILE)
print(f"Loaded {len(df_labels)} spine annotations.")
df_labels.head()

In [ ]:
def extract_vertex_features(root_id, vertex_indices):
    print(f"Processing Neuron {root_id}...")
    mesh = mm.mesh(seg_id=root_id)
    
    # Compute HKS for the WHOLE mesh
    # This takes ~1-2 minutes per neuron but we only do it once
    hks_result = condensed_hks_pipeline(
        (mesh.vertices.astype(np.float32), mesh.faces),
        verbose=True
    )
    
    # hks_result.labels maps the original mesh vertices to the 'condensed' HKS rows
    # We need to find the HKS features for your specific vertex_indices
    
    feature_table = hks_result.condensed_features
    mapping = hks_result.labels
    
    selected_features = []
    for v_idx in vertex_indices:
        row_idx = mapping[v_idx]
        if row_idx == -1:
            continue
        feat_val = feature_table.loc[row_idx].values
        selected_features.append(feat_val)
        
    return np.array(selected_features)

# Group by neuron and process
all_X = []
all_y = []

for root_id, group in df_labels.groupby('root_id'):
    v_indices = group['vertex_index'].values
    feats = extract_vertex_features(root_id, v_indices)
    
    all_X.append(feats)
    all_y.extend(['spine'] * len(feats))

X = np.vstack(all_X)
y = np.array(all_y)

print(f"\nCreated Feature Matrix X with shape {X.shape}")
print("This is ready for training!")